In [3]:
import Pkg

c = ["CSV", "Tidier", "DataFrames", "Metal", "RCall"]
Pkg.add(c)

using CSV, DataFrames, Tidier, Metal, RCall

# 座標データが時計回りかどうかを判定
function is_clockwise(df)
    n = size(df, 1)
    sum = 0.0
    for i in 1:n
        x0 = df.X[i]
        y0 = df.Y[i]
        x1 = df.X[mod1(i + 1, n)]
        y1 = df.Y[mod1(i + 1, n)]
        sum += (x1 - x0) * (y1 + y0)
    end

    return sum > 0.0

end

# PolygonIDごとに座標データを取り出し，反時計回りになるようにする
function to_counterclockwise(df)

    if is_clockwise(df)
        df = reverse(df)
        println("Clockwise")
    end

    return df

end

# CSVファイルから座標データを読み込む
function reading_coordinate()
    # CSVファイルを読み込む
    # polygon_df = CSV.read("../Plan/Polygon_Coordinate.csv", DataFrame)
    polygon_df = CSV.read("../Plan/25F_0205_base.csv", DataFrame)

    # PolygonIDごとに座標データを取り出し，反時計回りになるようにする
    ccw_df = DataFrame(
        X = Float32[],
        Y = Float32[],
        PolygonID = Int32[]
    )
    for i in unique(polygon_df.PolygonID)
        ccw_df = @chain polygon_df begin
            @filter(PolygonID == !!i)
            to_counterclockwise(_)
            vcat(ccw_df, _)
        end
    end
    polygon_df = ccw_df

    # 外周のポリゴンはPolygonIDが最小のポリゴンと仮定
    # PolygonIDが最小となる行を抽出
    # 変数Xと変数Yを取得
    minID = minimum(polygon_df.PolygonID)
    outer_df = @chain polygon_df begin
        @filter(PolygonID == !!minID)
        @rename(x0 = X, y0 = Y, id = PolygonID)
        @mutate(id = Int32(id + 1))
    end

    obs_df = @chain polygon_df begin
        @filter(PolygonID != !!minID)
        @rename(x0 = X, y0 = Y, id = PolygonID)
        @mutate(id = Int32(id + 1))
    end

    return (outer_df, obs_df)

end

# 外周および遮蔽物を構成する
# 辺を取得する
# 始点 (x0, y0）
# 終点 (x1, y1)
disassemble_poly2 = function (df_poly)
    edge_df = DataFrame(
        x0 = Float32[], y0 = Float32[], 
        x1 = Float32[], y1 = Float32[],
        id = Int32[])

    edge_df = @chain df_poly begin
        groupby(:id)
        transform(
            :x0 => (x -> circshift(x, -1)) => :x1,
            :y0 => (y -> circshift(y, -1)) => :y1
        )
        end
    
    # for i in  unique(df_poly.id)
    #     tmp_df = filter(row -> row.id == i, df_poly)
    #     n = size(tmp_df, 1)
    #     tmp_df.x1 = circshift(tmp_df.x0, -1)
    #     tmp_df.y1 = circshift(tmp_df.y0, -1)
    #     # tmp_df =
    #     # @chain df_poly begin
    #     #     @filter(id == i)
    #     #     @mutate(
    #     #         x1 = circshift(x0, -1),
    #     #         y1 = circshift(y0, -1)
    #     #     )
    #     # end
    #     append!(edge_df, tmp_df)
    # end

    return(edge_df)

end

# 内外判定
in_out_judge = function(xx, yy, poly_edge)
    n_x = length(xx)
    n_y = length(yy)

    poly_ids = unique(poly_edge.id)

    in_out = Array{Bool}(undef, n_x, n_y, length(poly_ids))
    in_out_df = DataFrame(
        index_x = (Int32)[],
        index_y = (Int32)[],
        # cal = (Bool)[]
    )

    for p in poly_ids
        edge =
        @chain poly_edge begin
            @filter(id == !!p)
            @select(-id)
        end

        n_e = nrow(edge)
    # @show edge
        winding_angle = zeros(Float32, n_x, n_y)

        vec0_x = Array{Float32}(undef, n_x, n_y, n_e)
        vec0_y = Array{Float32}(undef, n_x, n_y, n_e)
    
        vec1_x = Array{Float32}(undef, n_x, n_y, n_e)
        vec1_y = Array{Float32}(undef, n_x, n_y, n_e)
    
        for j in 1:n_e
            for i in 1:n_x
                @. vec0_x[i, :, j] = edge.x0[j] - xx[i]
                @. vec1_x[i, :, j] = edge.x1[j] - xx[i]
            end
            for i in 1:n_y
                @. vec0_y[:, i, j] = edge.y0[j] - yy[i]
                @. vec1_y[:, i, j] = edge.y1[j] - yy[i]
            end
        end
    # @show vec0_x <- OK
    # @show vec1_y <- OK

        Mtl_vec0_x = MtlArray(vec0_x)
        Mtl_vec0_y = MtlArray(vec0_y)
    
        Mtl_vec1_x = MtlArray(vec1_x)
        Mtl_vec1_y = MtlArray(vec1_y)
    
        # 外積
        Mtl_cross = similar(Mtl_vec0_x)
        @. Mtl_cross = Mtl_vec0_x * Mtl_vec1_y - Mtl_vec0_y * Mtl_vec1_x # OK

        # 2つのベクトルのなす角
        Mtl_dot = similar(Mtl_vec0_x)
        Mtl_norm_0 = similar(Mtl_vec0_x)
        Mtl_norm_1 = similar(Mtl_vec0_x)
        Mtl_norm = similar(Mtl_vec0_x)
        Mtl_angle = similar(Mtl_vec0_x)
        @. Mtl_dot = Mtl_vec0_x * Mtl_vec1_x + Mtl_vec0_y * Mtl_vec1_y
        @. Mtl_norm_0 = sqrt(Mtl_vec0_x * Mtl_vec0_x + Mtl_vec0_y * Mtl_vec0_y)
        @. Mtl_norm_1 = sqrt(Mtl_vec1_x * Mtl_vec1_x + Mtl_vec1_y * Mtl_vec1_y)
        @. Mtl_norm = Mtl_norm_0 * Mtl_norm_1
        # acos(dot / norm): dot / norm > 1 => NaN

        # 要素の値が1以上の時は1に置換する
        @chain Array(Mtl_dot ./ Mtl_norm) begin
            div = _
           _[_ .> 1] .= 1
        end
        Mtl_div = MtlArray(div)
        @. Mtl_angle = acos(Mtl_div)

    # dot = Array(Mtl_dot)
    # @show dot[148, 51, :]
    # norm = Array(Mtl_norm)
    # @show norm[148, 51, :]
    # angle = Array(Mtl_angle)
    # @show angle[148, 51, :]
        
    # angle_arr = Array(Mtl_angle)
    # angle_deg = similar(angle_arr)
    # @. angle_deg = angle_arr * (180.0 / π)
    # @show angle_deg[2, 2, :]
    # https://www.omnicalculator.com/math/angle-between-two-vectors
        Mtl_theta = similar(Mtl_angle)
        @. Mtl_theta = sign(Mtl_cross) * Mtl_angle # OK
        theta = Array(Mtl_theta)
        # angle = πのときtheta = 0.0となるときがある 原因不明
    # @show theta[148, 51, :]
        winding_angle[:, :] = sum(theta[:, :, i] for i in 1:n_e) # OK
    # @show winding_angle[148, 51]

    # https://www.nttpc.co.jp/technology/number_algorithm.html
    # INSIDEの時，winding_angle >- 2π
    # OUTSIDEの時, winding_angle == 0
        if p == 1
            # INSIDEが1
            in_out[:, :, p] = (winding_angle[:, :] .>=  1.99 * π)
            # in_out[:, :, p] = (winding_angle[:, :] .> 0.0)
        else
            # OUTSIDEが1
            # in_out[:, :, p] = (winding_angle[:, :] .< 0.1)
            in_out[:, :, p] = (winding_angle[:, :] .< 1.1 * π)
            # in_out[:, :, p] = (winding_angle[:, :] .<= 0.0)
        end
    end
    # @show in_out[148, 51, :]

    subject_cal_arr = Array{Bool}(undef, n_x, n_y)
    subject_cal_arr .= prod(in_out, dims = 3) # OK
    # @show subject_cal_arr[50, 80]

    df_y = repeat(Vector{Int32}(1:n_y), inner = [n_x])
    df_x = repeat(Vector{Int32}(1:n_x), outer = [n_y])

    in_out_df =
    @chain subject_cal_arr begin
        DataFrame(:auto)
        @pivot_longer(starts_with("x"), names_to = y, values_to = cal)
        insertcols!(:index_x => df_x)
        insertcols!(:index_y => df_y)
        @select(-y)
    end

    return in_out_df

end

# # 視線距離計算
# # Distance of Light of sight
# los_dist = function (all_points_df, edge_df)
#     # 半直線方向ベクトル用定数
#     # ll = convert(Float32, 99)
#     ll = Float32(99)

#     # スキャン角分割数
#     div_n = 50

#     cos_th =
#     @chain collect(1:div_n) begin
#         (_ .- 1) .* (2 * π / div_n)
#         cos.()
#         convert.(Float32, _)
#     end

#     sin_th =
#     @chain collect(1:div_n) begin
#         (_ .- 1) .* (2 * π / div_n)
#         sin.()
#         convert.(Float32, _)
#     end

#     # 外周および遮蔽物を構成する辺の数
#     num_edges = nrow(edge_df)
#     x0 = Vector{Float32}(edge_df.x0) # OK
#     y0 = Vector{Float32}(edge_df.y0)
#     x1 = Vector{Float32}(edge_df.x1) # 0K
#     y1 = Vector{Float32}(edge_df.y1)

#     # 計算実施点を取り出す
#     not_subject_df =
#     @chain all_points_df begin
#         @filter(cal != 1)
#     end

#     subject_df =
#     @chain all_points_df begin
#         @filter(cal == 1)
#     end

#     xx = Vector{Float32}(subject_df.xx) # OK
#     yy = Vector{Float32}(subject_df.yy) # OK
#     num_points = size(xx, 1)

#     # [点, 角度, 辺]
#     ab_x = Array{Float32}(undef, num_points, div_n, num_edges)
#     ab_y = similar(ab_x)
#     ac_x = similar(ab_x)
#     ac_y = similar(ab_x)
#     cd_x = similar(ab_x)
#     cd_y = similar(ab_x)
#     ca_x = similar(ab_x)
#     ca_y = similar(ab_x)
#     # ab_y = Array{Float32}(undef, num_points, div_n, num_edges)
#     # ac_x = Array{Float32}(undef, num_points, div_n, num_edges)
#     # ac_y = Array{Float32}(undef, num_points, div_n, num_edges)
#     # cd_x = Array{Float32}(undef, num_points, div_n, num_edges)
#     # cd_y = Array{Float32}(undef, num_points, div_n, num_edges)
#     # ca_x = Array{Float32}(undef, num_points, div_n, num_edges)
#     # ca_y = Array{Float32}(undef, num_points, div_n, num_edges)
#     # ab_x = ac_x = cd_x = ca_x =  Array{Float32}(undef, num_points, div_n, num_edges)
#     # ab_y = ac_y = cd_y = ca_y =  Array{Float32}(undef, num_points, div_n, num_edges)


#     # ab
#     for j in 1:div_n
#         @. ab_x[:, j, :] = ll * cos_th[j] # OK
#         @. ab_y[:, j, :] = ll * sin_th[j] # OK
#     end
#     Mtl_ab_x = MtlArray(ab_x)
#     Mtl_ab_y = MtlArray(ab_y)

#     # ac, cd
#     for k in 1:num_edges
#         @. cd_x[:, :, k] = x1[k] - x0[k] # OK
#         @. cd_y[:, :, k] = y1[k] - y0[k]
#         for i in 1:num_points
#             @. ac_x[i, :, k] = x0[k] - xx[i]
#             @. ac_y[i, :, k] = y0[k] - yy[i]
#         end
#     end

#     Mtl_ac_x = MtlArray(ac_x)
#     Mtl_ac_y = MtlArray(ac_y)
#     Mtl_cd_x = MtlArray(cd_x)
#     Mtl_cd_y = MtlArray(cd_y)

#     # ca
#     ca_x = -ac_x
#     ca_y = -ac_y
#     Mtl_ca_x = MtlArray(ca_x)
#     Mtl_ca_y = MtlArray(ca_y)
#     # Mtl_ca_x = -Mtl_ac_x
#     # Mtl_ca_y = -Mtl_ac_y

#     # 交点を示すパラメータ(s, t)
#     Mtl_s = similar(Mtl_ab_x)
#     Mtl_t = similar(Mtl_ab_x)
#     @. Mtl_s = (Mtl_ac_x * Mtl_cd_y - Mtl_ac_y * Mtl_cd_x) / (Mtl_ab_x * Mtl_cd_y - Mtl_ab_y * Mtl_cd_x)
#     @. Mtl_t = (Mtl_ca_x * Mtl_ab_y - Mtl_ca_y * Mtl_ab_x) / (Mtl_cd_x * Mtl_ab_y - Mtl_cd_y * Mtl_ab_x)
#     # 視線ベクトルと辺が平行な時，外積は0になる

#     # 有効な交点との距離
#     Mtl_s_valid = similar(Mtl_s)
#     Mtl_t_valid = similar(Mtl_t)
#     Mtl_dist_crossing = similar(Mtl_s)
#     # dist_crossing = fill(ll, num_points, div_n, num_edges) # 各辺との交点までの距離
#     # Mtl_dist_crossing = similar(dist_crossing) 3行下でエラー
#     @. Mtl_s_valid = (Mtl_s >= 0 && Mtl_s <= 1)
#     @. Mtl_t_valid = (Mtl_t >= 0 && Mtl_t <= 1)
#     @. Mtl_dist_crossing = ll * Mtl_s * Mtl_s_valid * Mtl_t_valid   # 平行 -> NaN 平行ではなく交点を持たない -> 0
#     # normal_s = Array(Mtl_s)
#     # normal_t = Array(Mtl_t)
#     # # @show xx[1], yy[1]
#     # @show ab_x[1, 1, 4], ab_y[1, 1, 4]
#     # @show cd_x[1, 1, 4], cd_y[1, 1, 4]
#     # @show ac_x[1, 1, 4], ac_y[1, 1, 4]
#     # @show ca_x[1, 1, 4], ca_y[1, 1, 4]
#     # @show normal_s[1, 1, 4]
#     # @show normal_t[1, 1, 4]

#     # 計算実施点から遮蔽物（外周）までの距離
#     # 計算実施点は必ずll未満の視線距離を有するので
#     # 視線と考慮対象エッジが平行な場合(NaN)および交点を持たない場合(0 or -0)は
#     # llとする ← 最小距離にはならないはず
#     min_dist_crossing =
#     @chain Array(Mtl_dist_crossing) begin
#         # @aside @show _[1, 4, :]
#         replace(0.0 => ll)
#         replace(-0.0 => ll)
#         replace(NaN => ll)
#         minimum(dims = 3)
#         reshape(num_points, div_n)
#     end

#     dist_df = DataFrame(
#         xx = xx,
#         yy = yy
#     )
    
# end

# t分布の確率密度関数 (自由度ν=1.1915889, 位置母数μ=0.1697106, 尺度母数σ=2.2166964)
# 低減回帰関数
function f(x)
    ν = 1.1915889f0
    μ = 0.1697106f0
    σ = 2.2166964f0
    t = (x - μ) / σ
    return (1 + t^2 / ν)^(-(ν + 1) / 2)

end
# カーネル関数（数値積分: 台形公式）
function kernel_integral(x0, y, steps, d)
    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y
    
    a = x0[i, j]
    
    integral = 0.0f0
    for k in 0:steps[i, j] - 1
        x1 = a + k * d
        x2 = a + (k + 1) * d
        integral += (f(x1) + f(x2)) / 2 * d
    end
    
    y[i, j] = integral
    
    return nothing
end

# 指標値計算
indeces = function (all_points_df, edge_df)
    # 半直線方向ベクトル用定数
    # ll = convert(Float32, 99)
    ll = Float32(99)

    # スキャン角分割数
    # div_n = 2000
    div_n = 200

    cos_th =
    @chain collect(1:div_n) begin
        (_ .- 1) .* (2 * π / div_n)
        cos.()
        convert.(Float32, _)
    end

    sin_th =
    @chain collect(1:div_n) begin
        (_ .- 1) .* (2 * π / div_n)
        sin.()
        convert.(Float32, _)
    end

    # 外周および遮蔽物を構成する辺の数
    num_edges = nrow(edge_df)
    x0 = Vector{Float32}(edge_df.x0) # OK
    y0 = Vector{Float32}(edge_df.y0)
    x1 = Vector{Float32}(edge_df.x1) # 0K
    y1 = Vector{Float32}(edge_df.y1)

    # 計算実施点を取り出す
    not_subject_df =
    @chain all_points_df begin
        @filter(cal != 1)
    end

    subject_df =
    @chain all_points_df begin
        @filter(cal == 1)
    end

    xx = Vector{Float32}(subject_df.xx) # OK
    yy = Vector{Float32}(subject_df.yy) # OK
    num_points = size(xx, 1)

    # [点, 角度, 辺]
    ab_x = Array{Float32}(undef, num_points, div_n, num_edges)
    ab_y = similar(ab_x)
    ac_x = similar(ab_x)
    ac_y = similar(ab_x)
    cd_x = similar(ab_x)
    cd_y = similar(ab_x)
    ca_x = similar(ab_x)
    ca_y = similar(ab_x)

    # ab
    # ab = bo - ao
    # bo = (ll * cos(θ) + xx, ll * sin(θ) + yy)
    # ao = (xx, yy)
    for j in 1:div_n
        @. ab_x[:, j, :] = ll * cos_th[j] # OK
        @. ab_y[:, j, :] = ll * sin_th[j] # OK
    end

    Mtl_ab_x = MtlArray(ab_x)
    Mtl_ab_y = MtlArray(ab_y)

    # ac, cd
    for k in 1:num_edges
        @. cd_x[:, :, k] = x1[k] - x0[k] # OK
        @. cd_y[:, :, k] = y1[k] - y0[k]
        for i in 1:num_points
            @. ac_x[i, :, k] = x0[k] - xx[i]
            @. ac_y[i, :, k] = y0[k] - yy[i]
        end
    end

    Mtl_ac_x = MtlArray(ac_x)
    Mtl_ac_y = MtlArray(ac_y)
    Mtl_cd_x = MtlArray(cd_x)
    Mtl_cd_y = MtlArray(cd_y)

    # ca
    ca_x = -ac_x
    ca_y = -ac_y
    Mtl_ca_x = MtlArray(ca_x)
    Mtl_ca_y = MtlArray(ca_y)
    # Mtl_ca_x = -Mtl_ac_x
    # Mtl_ca_y = -Mtl_ac_y

    # 交点を示すパラメータ(s, t)
    Mtl_s = similar(Mtl_ab_x)
    Mtl_t = similar(Mtl_ab_x)
    @. Mtl_s = (Mtl_ac_x * Mtl_cd_y - Mtl_ac_y * Mtl_cd_x) / (Mtl_ab_x * Mtl_cd_y - Mtl_ab_y * Mtl_cd_x)
    @. Mtl_t = (Mtl_ca_x * Mtl_ab_y - Mtl_ca_y * Mtl_ab_x) / (Mtl_cd_x * Mtl_ab_y - Mtl_cd_y * Mtl_ab_x)
    # 視線ベクトルと辺が平行な時，外積は0になる

    # 有効な交点との距離
    Mtl_s_valid = similar(Mtl_s)
    Mtl_t_valid = similar(Mtl_t)
    Mtl_dist_crossing = similar(Mtl_s)
    # dist_crossing = fill(ll, num_points, div_n, num_edges) # 各辺との交点までの距離
    # Mtl_dist_crossing = similar(dist_crossing) 3行下でエラー
    @. Mtl_s_valid = (Mtl_s >= 0 && Mtl_s <= 1)
    @. Mtl_t_valid = (Mtl_t >= 0 && Mtl_t <= 1)
    @. Mtl_dist_crossing = ll * Mtl_s * Mtl_s_valid * Mtl_t_valid   # 平行 -> NaN 平行ではなく交点を持たない -> 0
    # normal_s = Array(Mtl_s)
    # normal_t = Array(Mtl_t)
    # # @show xx[1], yy[1]
    # @show ab_x[1, 1, 4], ab_y[1, 1, 4]
    # @show cd_x[1, 1, 4], cd_y[1, 1, 4]
    # @show ac_x[1, 1, 4], ac_y[1, 1, 4]
    # @show ca_x[1, 1, 4], ca_y[1, 1, 4]
    # @show normal_s[1, 1, 4]
    # @show normal_t[1, 1, 4]

    # 計算実施点から遮蔽物（外周）までの距離
    # 計算実施点は必ずll未満の視線距離を有するので
    # 視線と考慮対象エッジが平行な場合(NaN)および交点を持たない場合(0 or -0)は
    # llとする ← 最小距離にはならないはず
    # Mtl_dist_crossing: 考慮点番号, 角度，辺番号
    # Mtl_min_dist_crossing: 考慮点番号, 角度
    Mtl_min_dist_crossing =
    @chain Array(Mtl_dist_crossing) begin
        # @aside @show _[1, 4, :]
        replace(0.0 => ll)
        replace(-0.0 => ll)
        replace(NaN => ll)
        minimum(dims = 3)
        reshape(num_points, div_n)
        MtlArray()
    end

    # A-Index用に低減関数で調整した距離を求める
    Δ = Float32(0.5f0)
    # Δ = 0.001f0
    Mtl_steps =MtlArray{Int32}(undef, num_points, div_n)
    @. Mtl_steps = unsafe_trunc(Int32, Mtl_min_dist_crossing / Δ)
    Mtl_start_p = Metal.zeros(Float32, num_points, div_n)

    Mtl_adj_dist = similar(Mtl_min_dist_crossing)
    threads_per_group = (32, 32)
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    ## カーネル関数（数値積分）を実行
    @metal threads = threads_per_group groups = num_groups kernel_integral(Mtl_start_p, Mtl_adj_dist, Mtl_steps, Δ)

    xx_arr = Array{Float32}(undef, num_points, div_n)
    yy_arr = similar(xx_arr)
    cos_arr = similar(xx_arr)
    sin_arr = similar(xx_arr)
    # yy_arr = Array{Float32}(undef, num_points, div_n)
    # cos_arr = Array{Float32}(undef, num_points, div_n)
    # sin_arr = Array{Float32}(undef, num_points, div_n)
    for i in 1:num_points
        @. xx_arr[i, :] = xx[i]
        @. yy_arr[i, :] = yy[i]
    end
    for j in 1:div_n
        @. cos_arr[:, j] = cos_th[j]
        @. sin_arr[:, j] = sin_th[j]
    end
    Mtl_xx = MtlArray(xx_arr)
    Mtl_yy = MtlArray(yy_arr)
    Mtl_cos = MtlArray(cos_arr)
    Mtl_sin = MtlArray(sin_arr)

    name_th = Vector{String}(undef, div_n)
    # 考慮点番号, 角度 1:(div_n - 1)
    for i in 1:div_n
        name_th[i] = "th_x_" * string(i - 1)
    end

    crossing_x_df =
    @chain similar(Mtl_min_dist_crossing) begin
        @. _ = Mtl_min_dist_crossing * Mtl_cos + Mtl_xx
        # # transpose()
        Array(_)
        DataFrame(name_th)
        @mutate(XX = !!xx)
    end

    crossing_adj_x_df =
    @chain similar(Mtl_adj_dist) begin
        @. _ = Mtl_adj_dist * Mtl_cos + Mtl_xx
        # # transpose()
        Array(_)
        DataFrame(name_th)
        @mutate(XX = !!xx)
    end

    # 考慮店番号, 角度 1:(div_n - 1)
    for i in 1:div_n
        name_th[i] = "th_y_" * string(i - 1)
    end

    crossing_y_df =
    @chain similar(Mtl_min_dist_crossing) begin
        @. _ = Mtl_min_dist_crossing * Mtl_sin + Mtl_yy
        # # transpose()
        Array(_)
        DataFrame(name_th)
        @mutate(YY = !!yy)
    end

    crossing_adj_y_df =
    @chain similar(Mtl_adj_dist) begin
        @. _ = Mtl_adj_dist * Mtl_sin + Mtl_yy
        # # transpose()
        Array(_)
        DataFrame(name_th)
        @mutate(YY = !!yy)
    end

    # Mtl_Isvst = MtlArray{Float32}(undef, num_points)
    # threads_per_group = (32, 32)
    # num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    # ## カーネル関数（座標法）を実行
    # @metal threads = threads_per_group groups = num_groups kernel_integral(Mtl_start_p, Mtl_adj_dist, Mtl_steps, Δ)

    # Mtl_crossing_df = hcat(Mtl_crossing_x_df, Mtl_crossing_y_df)
    # Mtl_crossing_adj_df = hcat(Mtl_crossing_adj_x_df, Mtl_crossing_adj_y_df)
    return (crossing_x_df, crossing_y_df,
            crossing_adj_x_df, crossing_adj_y_df)
    # Mtl_crossing_x = similar(Mtl_min_dist_crossing)
    # Mtl_crossing_y = similar(Mtl_min_dist_crossing)
    # @. Mtl_crossing_x = Mtl_min_dist_crossing * Mtl_cos
    # @. Mtl_crossing_y = Mtl_min_dist_crossing * Mtl_sin
    
    # @show (min_dist_crossing[1, 5], size(min_dist_crossing))
    # 多角形の面積の計算
    # PolygonOps.area
    # https://docs.juliahub.com/PolygonOps/TUelr/0.1.2/
end

# 外周および遮蔽物の生成
outer, obs = reading_coordinate()
# outer = generate_outer()
# outer_id_max = maximum(outer.id)
xmin_outer = minimum(outer.x0)
xmax_outer = maximum(outer.x0)
ymin_outer = minimum(outer.y0)
ymax_outer = maximum(outer.y0)

# 遮蔽物生成
# obs = generate_obs(outer_id_max)

# 外周及び遮蔽物を構成する
# 辺を取得
edge_outer = disassemble_poly2(outer)
@show edge_outer
edge_obs = disassemble_poly2(obs)
@show edge_obs
edge_df = vcat(edge_outer, edge_obs)

# 計算座標間隔
# 空間分割数
s_div = 300
dd_x = (xmax_outer - xmin_outer) / s_div
dd_y = (ymax_outer - ymin_outer) / s_div
xx = Vector{Float32}(undef, s_div + 1)
yy = Vector{Float32}(undef, s_div + 1)
# xx = Vector{Float32}(undef, (Int32)((xmax_outer - xmin_outer) / dd_x + 1))
# yy = Vector{Float32}(undef, (Int32)((ymax_outer - ymin_outer) / dd_y + 1))
xx[:] = [xmin_outer + (i - 1) * dd_x for i in 1:length(xx)]
yy[:] = [ymin_outer + (i - 1) * dd_y for i in 1:length(yy)]

# 内外判定
# PolygonOps,inpolygon
# https://docs.juliahub.com/PolygonOps/TUelr/0.1.2/
subject_cal_df = 
@chain in_out_judge(xx, yy, edge_df) begin
    @mutate(
        xx = (!!xmin_outer + (index_x - 1) * !!dd_x),
        yy = (!!ymin_outer + (index_y - 1) * !!dd_y)
    )
end


# # 視線距離計算
# los_dist_df = DataFrame(
#     x = Float32[],
#     y = Float32[],
#     dist = Float32[]
# )
# # 指標値計算
# los_dist_df = los_dist(subject_cal_df, edge_df)

# @time index_df = indeces(subject_cal_df, edge_df)
# 指標格納データフレーム
index_df = DataFrame(
    x = Float32[], 
    y = Float32[], 
    isovist = Float32[], 
    a_index = Float32[]
)
(vertex_x_df, vertex_y_df, vertex_adj_x_df, vertex_adj_y_df) = indeces(subject_cal_df, edge_df)
tmp_df = vertex_adj_x_df
@show size(tmp_df)
@show names(tmp_df)
# @show tmp_df[1, :]
# @show tmp_df[2, :]
#= 
@rput subject_cal_df
@rput xmin_outer
@rput ymin_outer
@rput xmax_outer
@rput ymax_outer
R"
library(tidyverse)

subject_cal_df =
    subject_cal_df %>%
    mutate(
        cal = case_when(
            cal == \"FALSE\" ~ 0,
            cal == \"TRUE\" ~ 1
        )
    ) %>%
    write_csv(\"subject_cal_df.csv\")

# # p <- 
# #     subject_cal_df %>%
# #     ggplot(aes(x = xx, y = yy, z = cal)) +
# #     stat_contour_filled(bins = 15) +
# #     theme(aspect.ratio = (ymax_outer - ymin_outer) / (xmax_outer - xmin_outer))
# # plot(p)
" =#

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


edge_outer = 10×5 DataFrame
 Row │ x0       y0       id     x1       y1
     │ Float64  Float64  Int32  Float64  Float64
─────┼───────────────────────────────────────────
   1 │    44.8     30.9      1     28.0     30.9
   2 │    28.0     30.9      1     28.0     32.8
   3 │    28.0     32.8      1     19.2     32.8
   4 │    19.2     32.8      1     19.2     27.0
   5 │    19.2     27.0      1     16.8     27.0
   6 │    16.8     27.0      1     16.8     15.2
   7 │    16.8     15.2      1     16.0     15.2
   8 │    16.0     15.2      1     16.0     12.0
   9 │    16.0     12.0      1     44.8     12.0
  10 │    44.8     12.0      1     44.8     30.9
edge_obs = 0×5 DataFrame
 Row │ x0       y0       id     x1       y1
     │ Float64  Float64  Int32  Float64  Float64
─────┴───────────────────────────────────────────
size(tmp_df) = (80940, 201)
names(tmp_df) = ["th_x_0", "th_x_1", "th_x_2", "th_x_3", "th_x_4", "th_x_5", "th_x_6", "th_x_7", "th_x_8", "th_x_9", "th_x_10", "th_x_11", "th_

201-element Vector{String}:
 "th_x_0"
 "th_x_1"
 "th_x_2"
 "th_x_3"
 "th_x_4"
 "th_x_5"
 "th_x_6"
 "th_x_7"
 "th_x_8"
 "th_x_9"
 ⋮
 "th_x_192"
 "th_x_193"
 "th_x_194"
 "th_x_195"
 "th_x_196"
 "th_x_197"
 "th_x_198"
 "th_x_199"
 "XX"